# DLPFC Cross-Slice Generalization Classification

Evaluate PCA and PCA + image embeddings (GPFM / UNI / UNI2-h / GigaPath): 8 training slices -> 4 held-out test slices.


In [ ]:
import os, pickle, warnings, sys
from pathlib import Path

# Locate the standalone feature module from the working directory.
FEATURE_ROOT = next(
    (candidate for parent in (Path.cwd(), *Path.cwd().parents)
     for candidate in (parent, parent / "feature")
     if (candidate / "src" / "spaug_feature").is_dir()),
    None,
)
if FEATURE_ROOT is None:
    raise RuntimeError("Start the kernel from the feature module or its notebook directory.")
sys.path.insert(0, str(FEATURE_ROOT / "src"))

import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, matthews_corrcoef
import scanpy as sc
import torch
warnings.filterwarnings("ignore")

from spaug_feature import spatial as spg
from spaug_feature.classifiers import make_classifier as spg_make_classifier, train_classifier as spg_train_classifier
from spaug_feature.cross_slice import classifier_params as spg_classifier_params, transform_features as spg_transform_features, project_gpu_pca as spg_project_gpu_pca

print("Feature module imports ready")
from spaug_feature.alignment import align_observations, match_sample_barcodes


## Configuration

Parameters are read from `configs/cross_slice_generalization/DLPFC/downstream.yaml`.


In [ ]:
# ══════════════════════════════════════════════════════════
# Read parameters from the feature config file
# ══════════════════════════════════════════════════════════
cfg = spg.load_config("configs/cross_slice_generalization/DLPFC/downstream.yaml")
task_cfg = cfg["cross_slice_generalization"]

# All split_tags and their corresponding slice splits
SPLIT_TAGS = task_cfg.get("split_tags", ["fixed_8train_4test"])
SPLIT_SLICES = {}
for tag in SPLIT_TAGS:
    if tag == "fixed_8train_4test":
        SPLIT_SLICES[tag] = {
            "train": task_cfg["train_slices"],
            "test": task_cfg["test_slices"],
        }
    else:
        es = task_cfg["extra_splits"][tag]
        SPLIT_SLICES[tag] = {
            "train": es["train_slices"],
            "test": es["test_slices"],
        }

# Use the first split as the global ALL_SLICES by default (all slices are used for data loading)
TRAIN_SLICES = SPLIT_SLICES[SPLIT_TAGS[0]]["train"]
TEST_SLICES = SPLIT_SLICES[SPLIT_TAGS[0]]["test"]
ALL_SLICES = sorted(set(sum([list(v["train"]) + list(v["test"]) for v in SPLIT_SLICES.values()], [])))

# Classification hyperparameters
SEEDS = task_cfg.get("seeds", [task_cfg.get("random_seed", 42)])
PCA_N = task_cfg.get("pca_n_components", 50)
CLASSIFIER_NAMES = task_cfg.get("classifiers", ["LR", "XGBoost", "MLP"])

# Metric list
METRIC_CONFIG_NAMES = task_cfg.get("metrics", ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "mcc"])
METRIC_DISPLAY = {
    "accuracy": "Accuracy", "balanced_accuracy": "BalAcc",
    "macro_f1": "MacroF1", "weighted_f1": "WeightedF1", "mcc": "MCC",
}
METRIC_COLS = [METRIC_DISPLAY.get(m, m) for m in METRIC_CONFIG_NAMES]

# HVG parameters
hvg_cfg = spg.load_config("configs/common/DLPFC/normalize.yaml")
HVG_TOP_GENES = hvg_cfg.get("preprocessing", {}).get("DLPFC", {}).get("n_top_genes", 3000)

# Image embedding filename suffix
PKL_SUFFIX = "112"

# Data and output paths
DATA_DIR = (FEATURE_ROOT / "data/DLPFC")
OUTPUT_DIR = (FEATURE_ROOT / "outputs/cross_slice_comparison")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Config: configs/cross_slice_generalization/DLPFC/downstream.yaml")
print(f"  split_tags: {SPLIT_TAGS}")
for tag in SPLIT_TAGS:
    sl = SPLIT_SLICES[tag]
    print(f"    {tag}: train={sl['train']}, test={sl['test']}")
print(f"  PCA_N={PCA_N}, HVG_TOP_GENES={HVG_TOP_GENES}")
"print(f\"  Metrics: {METRIC_COLS}\")"

## Data Loading and Preprocessing

Use the module-local prepared h5ad (QC + normalize + HVG) and align the image embeddings.


In [ ]:
# (key, display_name, dim)
EMBED_CONFIG = [
    ("gpfm",     "GPFM",   1024),
    ("uni",      "UNI",    1024),
    ("uni2_h",   "UNI2-h", 1536),
    ("gigapath", "GigaPath", 1536),
]

print("Loading embeddings...")
emb_data = {}
for key, name, dim in EMBED_CONFIG:
    emb_data[key] = {}
    for sid in ALL_SLICES:
        p = DATA_DIR / "pkl" / "embeddings" / f"visium_{sid}_allspot_{key}_{PKL_SUFFIX}.pkl"
        emb_data[key][sid] = torch.load(p, weights_only=True, map_location="cpu").numpy().astype("float32")
    print(f"  {name:20s} ({dim:4d}d): {len(emb_data[key])} slices")

# ══════════════════════════════════════════════════════════
# Data loading: use the module-local prepared h5ad
# First load the combined file to get the global shared genes
# Then read slice by slice and align everything to that gene space
# ══════════════════════════════════════════════════════════
PROCESSED_DIR = (FEATURE_ROOT / "data/02_interim/common/DLPFC")

# Load the combined file to get the global shared gene space
combined = ad.read_h5ad(PROCESSED_DIR / "processed_combined.h5ad")
# Use all shared genes (uses the local feature projection)
COMMON_GENES = combined.var_names.tolist()
# To use global HVGs instead, change to: COMMON_GENES = combined.var_names[combined.var["highly_variable"]].tolist()
print(f"Global shared genes: {len(COMMON_GENES)}")

adatas = {}

for sid in ALL_SLICES:
    a = ad.read_h5ad(PROCESSED_DIR / f"{sid}_processed.h5ad")

    # Align embeddings: the processed h5ad obs_names carry a "{sid}_" prefix
    a_raw = ad.read_h5ad(DATA_DIR / "st" / f"{sid}_adata.h5ad")
    source_names = [f"{sid}_{name}" for name in a_raw.obs_names]
    for key, _, _ in EMBED_CONFIG:
        e = emb_data.get(key, {}).get(sid)
        if e is not None:
            emb_data[key][sid] = align_observations(e, source_names, a.obs_names)

    # Align the gene space using globally shared genes
    # Ensure gene alignment when vstacking across slices
    common = [g for g in COMMON_GENES if g in a.var_names]
    a = a[:, common].copy()
    # Unify the label column name
    lbl = a.obs["label"]
    a.obs["spatialLIBD"] = lbl.values if hasattr(lbl, "values") else lbl
    adatas[sid] = a

print(f"Loaded {len(adatas)} slices, using {len(COMMON_GENES)} global shared genes")


## Classifiers

LR / XGBoost / MLP from the feature-local `classifiers.py`.


In [ ]:
# ══════════════════════════════════════════════════════════
# Classifiers imported directly from the feature module:
#   spg_make_classifier(name, params, random_seed) — classifiers.py
#   spg_train_classifier(x_train, y_train, classifier, params, random_seed) — classifiers.py
#   spg_classifier_params(classifier, use_gpu, n_jobs) — predict.py
# ══════════════════════════════════════════════════════════

# Classifier parameters (identical to synthetic predict.py)
CLF_PARAMS = {
    name: spg_classifier_params(name, use_gpu="auto", n_jobs=4)
    for name in CLASSIFIER_NAMES
}

print("Helper functions ready")
print("  make_classifier / train_classifier → from classifiers.py")
print("  classifier_params → from predict.py")


## Feature Construction

PCA(50) baseline, and PCA(50) concatenated with each image embedding.


In [ ]:
def transform_pca_embedding_features(train_expr, test_expr, train_emb, test_emb, seed=42):
    """Project expression with the shared PCA implementation.

    Uses the task-local transform_features and project_gpu_pca to keep the
    decomposition and feature scaling consistent across feature experiments.
    """
    n = min(PCA_N, train_expr.shape[1], max(1, int(train_expr.shape[0]) - 1))
    if n < 2:
        return np.hstack([train_expr, train_emb]), np.hstack([test_expr, test_emb])
    tr = train_expr.toarray() if sparse.issparse(train_expr) else np.asarray(train_expr)
    te = test_expr.toarray() if sparse.issparse(test_expr) else np.asarray(test_expr)
    tr = np.asarray(tr, dtype=np.float32); te = np.asarray(te, dtype=np.float32)
    out = spg_project_gpu_pca(tr, tr, te, n, use_gpu="auto")
    if out is None:  # GPU unavailable: use the exact fallback equivalent to eigh
        pca = spg.PCA(n_components=n, random_state=42)  # uses the local PCA fallback
        pca.fit(tr)
        out = (pca.transform(tr), pca.transform(te))
    return np.hstack([out[0], train_emb]), np.hstack([out[1], test_emb])
print("Feature function definitions ready")
print("  transform_pca_features / transform_pca_embedding_features → match synthetic transform_features")


## Run the Cross-Slice Experiment

Train on the training slices, predict on the held-out slices, then save `predictions.csv` + `manifest.json`.


In [ ]:
import json

all_rows = []
pred_rows = []

for split_tag in SPLIT_TAGS:
    train_slices = SPLIT_SLICES[split_tag]["train"]
    test_slices = SPLIT_SLICES[split_tag]["test"]
    print(f"\n{'='*60}\n# split_tag: {split_tag}\n#   train: {train_slices}\n#   test:  {test_slices}\n{'='*60}")

    for clf_name in CLASSIFIER_NAMES:
        for seed in SEEDS:
            # ── Concatenate training slices ──
            train_data = {k: [] for k, _, _ in EMBED_CONFIG}
            expr_list, label_list = [], []
            train_obs_list = []
            for sid in train_slices:
                a = adatas[sid]
                keep = ~pd.isna(a.obs["spatialLIBD"])
                expr_list.append(a.X[keep.values])
                train_obs_list.append(a.obs_names[keep.values])
                for k, _, _ in EMBED_CONFIG:
                    train_data[k].append(emb_data[k][sid][keep.values])
                label_list.append(a.obs["spatialLIBD"].values[keep.values])

            train_expr = sparse.vstack(expr_list) if sparse.issparse(expr_list[0]) else np.vstack(expr_list)
            train_labels = np.concatenate(label_list)
            n_train_total = int(train_expr.shape[0])

            # ── Concatenate all test slices (prepared in advance, used for LabelEncoder + transform) ──
            test_expr_list, test_labels_list, test_obs_list = [], [], []
            test_embs_list = {k: [] for k, _, _ in EMBED_CONFIG}
            test_n_per_slice = {}
            for sid in test_slices:
                a = adatas[sid]
                keep = ~pd.isna(a.obs["spatialLIBD"])
                test_expr_list.append(a.X[keep.values])
                test_labels_list.append(a.obs["spatialLIBD"].values[keep.values])
                test_obs_list.append(a.obs_names[keep.values])
                for k, _, _ in EMBED_CONFIG:
                    test_embs_list[k].append(emb_data[k][sid][keep.values])
                test_n_per_slice[sid] = int(a.X[keep.values].shape[0])

            test_expr_all = sparse.vstack(test_expr_list) if sparse.issparse(test_expr_list[0]) else np.vstack(test_expr_list)
            test_adata_all = ad.AnnData(X=test_expr_all)

            # ── LabelEncoder (test_labels_list is now defined) ──
            _le = LabelEncoder()
            _le.fit(np.concatenate([train_labels] + test_labels_list))
            train_labels_enc = _le.transform(train_labels)

            # ── PCA baseline: train (matching synthetic) ──
            train_adata = ad.AnnData(X=train_expr)
            feats_pca, _ = spg_transform_features(
                train_adata, train_adata,
                n_components=PCA_N, use_gpu="auto",
            )
            params = CLF_PARAMS[clf_name]
            clf_pca = spg_train_classifier(feats_pca, train_labels_enc, clf_name, params, random_seed=seed)

            # ── Train for each embedding ──
            clf_emb = {}
            for k, _, _ in EMBED_CONFIG:
                feats_emb, _ = transform_pca_embedding_features(train_expr, train_expr[:1],
                                                                np.vstack(train_data[k]), np.vstack(train_data[k])[:1], seed)
                clf_emb[k] = spg_train_classifier(feats_emb, train_labels_enc, clf_name, params, random_seed=seed)

            # ── PCA: transform all test slices at once ──
            _, feats_test_all = spg_transform_features(
                train_adata, test_adata_all,
                n_components=PCA_N, use_gpu="auto",
            )
            pred_all = _le.inverse_transform(clf_pca.predict(feats_test_all))

            # ── PCA+embedding: transform at once ──
            pred_emb_all = {}
            for k, name, _ in EMBED_CONFIG:
                test_emb_all = np.vstack(test_embs_list[k])
                _, feats_emb_test_all = transform_pca_embedding_features(train_expr, test_expr_all,
                                                                         np.vstack(train_data[k]), test_emb_all, seed)
                pred_emb_all[k] = _le.inverse_transform(clf_emb[k].predict(feats_emb_test_all))

            # ── Evaluate by splitting per test slice ──
            offset = 0
            for si_idx, sid in enumerate(test_slices):
                n = test_n_per_slice[sid]
                test_labels = test_labels_list[si_idx]

                # PCA
                pred = pred_all[offset:offset+n]
                all_rows.append({"TestSlice": sid, "Classifier": clf_name, "Seed": seed,
                    "SplitTag": split_tag, "Method": "PCA",
                    "Accuracy": round(accuracy_score(test_labels, pred), 4),
                    "MacroF1": round(f1_score(test_labels, pred, average="macro"), 4),
                    "WeightedF1": round(f1_score(test_labels, pred, average="weighted"), 4),
                    "BalAcc": round(balanced_accuracy_score(test_labels, pred), 4),
                    "MCC": round(matthews_corrcoef(test_labels, pred), 4)})
                for obs_name, t, p in zip(test_obs_list[si_idx], test_labels, pred):
                    pred_rows.append({
                        "obs_name": str(obs_name),
                        "slice_id": str(sid),
                        "split_tag": split_tag,
                        "test_scope": "heldout_slice",
                        "y_true": str(t),
                        "y_pred": str(p),
                        "classifier": clf_name,
                        "model": "PCA",
                        "paradigm": "real_baseline",
                        "variant_id": "real_baseline",
                        "ratio": "",
                        "alpha": "",
                        "random_seed": int(seed),
                        "n_train_real_total": n_train_total,
                        "n_test_total": n,
                        "n_train_package": n_train_total,
                        "n_train_effective": n_train_total,
                        "n_train_slices": len(train_slices),
                        "n_test_slices": len(test_slices),
                        "local_spatial_ratio_role": "",
                    })

                # PCA + each embedding
                for k, name, _ in EMBED_CONFIG:
                    pred = pred_emb_all[k][offset:offset+n]
                    all_rows.append({"TestSlice": sid, "Classifier": clf_name, "Seed": seed,
                        "SplitTag": split_tag, "Method": f"PCA+{name}",
                        "Accuracy": round(accuracy_score(test_labels, pred), 4),
                        "MacroF1": round(f1_score(test_labels, pred, average="macro"), 4),
                        "WeightedF1": round(f1_score(test_labels, pred, average="weighted"), 4),
                        "BalAcc": round(balanced_accuracy_score(test_labels, pred), 4),
                        "MCC": round(matthews_corrcoef(test_labels, pred), 4)})
                    for obs_name, t, p in zip(test_obs_list[si_idx], test_labels, pred):
                        pred_rows.append({
                            "obs_name": str(obs_name),
                            "slice_id": str(sid),
                            "split_tag": split_tag,
                            "test_scope": "heldout_slice",
                            "y_true": str(t),
                            "y_pred": str(p),
                            "classifier": clf_name,
                            "model": f"PCA+{name}",
                            "paradigm": "real_baseline",
                            "variant_id": "real_baseline",
                            "ratio": "",
                            "alpha": "",
                            "random_seed": int(seed),
                            "n_train_real_total": n_train_total,
                            "n_test_total": n,
                            "n_train_package": n_train_total,
                            "n_train_effective": n_train_total,
                            "n_train_slices": len(train_slices),
                            "n_test_slices": len(test_slices),
                            "local_spatial_ratio_role": "",
                        })
                offset += n
            print(f"  {clf_name} seed={seed} done", flush=True)

df = pd.DataFrame(all_rows)
print(f"\n{len(df)} result rows in total, {len(pred_rows)} per-spot predictions")
print(f"  split_tags: {df['SplitTag'].unique()}")


In [ ]:
metric_cols = METRIC_COLS

print("="*100)
print("  Results for each split_tag + test slice")
print("="*100)
display(df.groupby(["SplitTag", "TestSlice", "Method", "Classifier"])[metric_cols].mean().round(4))

print("\n" + "="*100)
print("  Global summary (across test slices and seeds)")
print("="*100)
display(df.groupby(["SplitTag", "Method", "Classifier"])[metric_cols].agg(["mean", "std"]).round(4))

In [ ]:
# Delta table: gain / loss of each embedding relative to PCA
print("="*100)
print("  ΔAccuracy / ΔMacroF1 (each embedding - PCA), aggregated by split_tag")
print("="*100)
for tag in df["SplitTag"].unique():
    print(f"\n--- {tag} ---")
    sub = df[df["SplitTag"] == tag]
    for clf_name in CLASSIFIER_NAMES:
        pca = sub[(sub["Method"]=="PCA") & (sub["Classifier"]==clf_name)]
        if len(pca) == 0:
            continue
        baseline_acc = pca["Accuracy"].mean()
        baseline_f1 = pca["MacroF1"].mean()
        for method_name in [m for m in sub["Method"].unique() if m != "PCA"]:
            s = sub[(sub["Method"]==method_name) & (sub["Classifier"]==clf_name)]
            if len(s) == 0:
                continue
            da = s["Accuracy"].mean() - baseline_acc
            df1 = s["MacroF1"].mean() - baseline_f1
            print(f"  {method_name:20s}  {clf_name:8s}:  ΔAcc={da:+.4f}  ΔMacroF1={df1:+.4f}")

In [ ]:
import json

df_pred = pd.DataFrame(pred_rows)
df_pred.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
print(f"Per-spot predictions saved: {OUTPUT_DIR / 'predictions.csv'} ({len(df_pred)} rows)")

df.to_csv(OUTPUT_DIR / "per_run_results.csv", index=False)
df.groupby(["SplitTag", "Method", "Classifier", "TestSlice"])[metric_cols].agg(["mean", "std"]).round(4).to_csv(OUTPUT_DIR / "summary.csv")

# ══════════════════════════════════════════════════════════════
# Save in a synthetic-compatible format: one directory per (model, split_tag)
# Output structure:
#   cross_slice_generalization/DLPFC/{model}/{split_tag}/
#     predictions.csv (per-spot predictions for that method + split_tag)
#     manifest.json
# ══════════════════════════════════════════════════════════════
SPG_ROOT = (FEATURE_ROOT / "outputs/cross_slice_comparison")
n_written = 0

for split_tag in df["SplitTag"].unique():
    train_slices = SPLIT_SLICES[split_tag]["train"]
    test_slices = SPLIT_SLICES[split_tag]["test"]

    for method_name in df["Method"].unique():
        # model_key: PCA → "baseline", PCA+GPFM → "PCA_plus_GPFM"
        is_pca = (method_name == "PCA")
        model_key = "baseline" if is_pca else method_name.replace("+", "_plus_")

        out_dir = SPG_ROOT / "cross_slice_generalization" / "DLPFC" / model_key / split_tag
        out_dir.mkdir(parents=True, exist_ok=True)

        # ── predictions.csv ──
        mask = (df_pred["split_tag"] == split_tag) & (df_pred["model"] == method_name)
        n_pred = mask.sum()
        if n_pred:
            df_pred[mask].to_csv(out_dir / "predictions.csv", index=False)

        # ── manifest.json ──
        sub = df[(df["SplitTag"] == split_tag) & (df["Method"] == method_name)]
        quick_acc = round(float(sub["Accuracy"].mean()), 4) if len(sub) > 0 else 0.0

        manifest = {
            "dataset": "DLPFC",
            "task_family": "cross_slice_generalization",
            "task_name": "dlpfc_cross_slice_spot_clf",
            "model": model_key,
            "paradigm": "real_baseline",
            "variant_id": "baseline",
            "ratio": None,
            "alpha": None,
            "split_tag": split_tag,
            "synthetic_split_tag": "none",
            "synthetic_usage_policy": "none",
            "test_synthetic_used": False,
            "random_seed": 42,
            "prediction_scope": "cross_slice",
            "train_slices": list(train_slices),
            "test_slices": list(test_slices),
            "pca_fit_scope": "combined_train_package",
            "pca_components": PCA_N,
            "classifiers": CLASSIFIER_NAMES,
            "n_test_spots": int(n_pred),
            "quick_accuracy": quick_acc,
        }
        (out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        n_written += 1

print(f"\nResults saved to {OUTPUT_DIR.resolve()}/")
print(f"Wrote {n_written} (model, split_tag) directories, each with predictions.csv + manifest.json")
if n_written > 0:
    print("Directory listing:")
    for p in sorted(SPG_ROOT.glob("cross_slice_generalization/DLPFC/*/*/predictions.csv")):
        model_part = p.parent.parent.stem
        split_part = p.parent.stem
        print(f"  {model_part:25s} / {split_part}")